# Signac 100M — Kaggle free TPU qualification

This notebook targets Kaggle's free TPU runtime and requires eight devices. Automated receipts identify TPU device type and counts plus software/platform versions, but do not identify the chip SKU or Kaggle image build; record those from the session when reporting target-specific results. It runs bounded synthetic multi-device canaries for the Signac M102 model and two architecture challengers. Every model worker is launched through `torch_xla.launch`; each canary must account for all eight ranks and devices before it reports success.

These receipts qualify engineering plumbing only. They do not qualify production training, the research corpus, Citadel evaluation, production exact distributed restart, or an AGI capability claim. The optional M102 restart canary also saves and replays per-rank CPU and XLA RNG around a synthetic random probe; this is not a production sampler restart. See `docs/signac_100m/PHASE1_TWIN_CONTRACT.md` and `docs/signac_100m/KAGGLE_TPU_RUNBOOK.md`.


In [ ]:
from pathlib import Path
import subprocess, sys

candidates = [Path('/kaggle/working/An-Ra-the-new-AGI-1'), *Path('/kaggle/input').glob('*/An-Ra-the-new-AGI-1'), *[p.parent for p in Path('/kaggle/input').glob('*/signac_100m')]]
roots = [p.resolve() for p in candidates if (p / 'signac_100m' / 'spec.py').is_file()]
if len(roots) != 1:
    raise RuntimeError(f'Expected exactly one attached repository snapshot; found {roots}')
REPO = roots[0]
sys.path.insert(0, str(REPO))
from signac_100m.source_identity import build_source_identity
SOURCE_IDENTITY = build_source_identity(REPO)
COMMIT = SOURCE_IDENTITY['source_commit'] or 'UNAVAILABLE: Kaggle snapshot has no Git metadata'
print('Repository:', REPO)
print('Commit:', COMMIT)
print('Source tree SHA-256:', SOURCE_IDENTITY['source_tree_sha256'])


In [ ]:
import importlib.metadata as md
import json, platform, torch

runtime = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'torch_xla': md.version('torch_xla'),
    'platform': platform.platform(),
    'accelerator_requested_in_kaggle_ui': 'TPU',
    'target_global_devices': 8,
    'source_commit': SOURCE_IDENTITY['source_commit'],
    'source_tree_sha256': SOURCE_IDENTITY['source_tree_sha256'],
}
print(json.dumps(runtime, indent=2))
Path('/kaggle/working/signac_100m_runtime.json').write_text(
    json.dumps(runtime, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)
assert runtime['torch_xla'], 'Use the Kaggle-provided TPU image; do not silently replace its matched torch_xla runtime.'
# Do not call xm.xla_device() or initialize any XLA tensor in the notebook
# parent. The distributed worker module discovers topology after torch_xla.launch.


In [ ]:
from signac_100m.spec import (
    MODEL_SPEC, TPU_DEPTH_PRESERVING_CHALLENGER, TPU_TILED_CHALLENGER, candidate_receipts,
)
from tools.signac_100m_preflight import build_report

receipts = candidate_receipts()
print(json.dumps({name: {
    'parameters': row['resources']['parameters_exact'],
    'model_spec_sha256': row['model_spec_sha256'],
    'attention_score_tensor_bytes_bf16_per_replica_per_active_layer': row['resources']['attention_score_tensor_bytes_bf16_per_replica_per_active_layer'],
} for name, row in receipts.items()}, indent=2))
assert MODEL_SPEC.parameter_receipt().total == 101_790_080
assert TPU_DEPTH_PRESERVING_CHALLENGER.parameter_receipt().total == 99_332_480
assert TPU_TILED_CHALLENGER.parameter_receipt().total == 100_303_104
static_report = build_report(target='tpu')
static_report['source_identity'] = SOURCE_IDENTITY
Path('/kaggle/working/signac_100m_source_identity.json').write_text(
    json.dumps(SOURCE_IDENTITY, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)
Path('/kaggle/working/signac_100m_static_preflight.json').write_text(
    json.dumps(static_report, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)
print('Static launch-gate report:', json.dumps({
    'verdict': static_report['verdict'],
    'blockers': static_report['blockers'],
    'training_authorized': static_report['training_authorized'],
}, indent=2))
assert static_report['training_authorized'] is False


## All-core TPU update canaries

The next cell launches one fresh worker group per candidate. Each worker uses a deterministic, rank-specific synthetic batch, four 4,096-token microsteps per update, BF16 autocast with FP32 weights and Adam state, averaged gradients followed by one global clip, and two synchronized optimizer updates. The first update includes compilation; the second is a bounded steady-state sample. For M102, `--verify-restart` also starts a fresh worker group and replays the saved XLA random probe, CPU/XLA RNG state, and synthetic stream cursor. Set `RUN_M102_QUALIFICATION = True` in the code cell to request a separate 21-update M102 profile with 20 measured steady updates and a fail-closed 85% TPU device-memory peak gate. Set `RUN_XLA_DEVELOPMENT_CAMPAIGN = True` to run the actual bounded production-backend campaign with synthetic data: one update, a fresh worker group, then exact checkpoint resume for update two. That lane is unqualified engineering evidence only and is disabled by default. Rank receipts record per-worker Linux process-lifetime high-water RSS after each update and after final parameter/Adam hashes when available; it is diagnostic and does not replace that gate. The final sample covers only the canary end-of-run hashes; production hashes every update. Rank-zero is not a special writer for the per-rank receipts; the bounded restart check writes one replicated model/Adam checkpoint.


In [ ]:
import os, time
RUN_M102_QUALIFICATION = False  # Set True to measure 20 steady updates and peak memory.
RUN_XLA_DEVELOPMENT_CAMPAIGN = False  # Synthetic only; exercise production-backend XLA wiring and resume.

candidates = [
    'm102_primary',
    'tpu_depth_preserving_challenger',
    'tpu_tiled_challenger',
]
run_id = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + '_' + str(time.time_ns())[-9:]
receipt_root = Path('/kaggle/working/signac_100m_distributed') / run_id
child_env = os.environ.copy()
child_env['PYTHONPATH'] = str(REPO) + os.pathsep + child_env.get('PYTHONPATH', '')
all_core_receipts = {'source_identity': SOURCE_IDENTITY, 'candidates': {}}
for candidate in candidates:
    command = [
        sys.executable, '-m', 'v5_training.kaggle_tpu_canary',
        '--source-tree-sha256', SOURCE_IDENTITY['source_tree_sha256'],
        '--candidate', candidate,
        '--output-dir', str(receipt_root),
        '--expected-global-devices', '8',
        '--expected-world-size', '8',
        '--sequence-length', '4096',
        '--microbatch-size', '1',
        '--accumulation-steps', '4',
        '--optimizer-updates', '2',
        '--minimum-steady-updates', '1',
        '--seed', '73011',
    ]
    if candidate == 'm102_primary':
        command.append('--verify-restart')
    print('Starting all-core canary:', candidate, flush=True)
    completed = subprocess.run(command, cwd=str(REPO), env=child_env, check=False)
    if completed.returncode:
        raise RuntimeError(f'{candidate} all-core Kaggle TPU canary failed ({completed.returncode}); retain rank failure receipts and notebook output.')
    aggregate_path = receipt_root / candidate / 'aggregate.json'
    aggregate = json.loads(aggregate_path.read_text(encoding='utf-8'))
    assert aggregate['status'] == 'PASS'
    assert aggregate['aggregate']['participating_ordinals'] == list(range(8))
    all_core_receipts['candidates'][candidate] = aggregate
    print(candidate, json.dumps(aggregate['aggregate'], indent=2))
if RUN_M102_QUALIFICATION:
    qualification_root = receipt_root / 'qualification'
    qualification_command = [
        sys.executable, '-m', 'v5_training.kaggle_tpu_canary',
        '--source-tree-sha256', SOURCE_IDENTITY['source_tree_sha256'],
        '--candidate', 'm102_primary',
        '--output-dir', str(qualification_root),
        '--expected-global-devices', '8',
        '--expected-world-size', '8',
        '--sequence-length', '4096',
        '--microbatch-size', '1',
        '--accumulation-steps', '4',
        '--optimizer-updates', '21',
        '--minimum-steady-updates', '20',
        '--seed', '73011',
    ]
    print('Starting 20-update M102 qualification profile', flush=True)
    completed = subprocess.run(qualification_command, cwd=str(REPO), env=child_env, check=False)
    if completed.returncode:
        raise RuntimeError('20-update M102 qualification failed; retain the rank receipts and TPU output.')
    qualification_path = qualification_root / 'm102_primary' / 'aggregate.json'
    qualification = json.loads(qualification_path.read_text(encoding='utf-8'))
    if qualification['aggregate']['measurement_profile'] != 'QUALIFICATION':
        raise RuntimeError('M102 qualification receipt did not pass its profile gate.')
    all_core_receipts['m102_qualification'] = qualification
if RUN_XLA_DEVELOPMENT_CAMPAIGN:
    development_root = receipt_root / 'production_backend_xla_development'
    development_command = [
        sys.executable, '-m', 'v5_training.kaggle_xla_development',
        '--source-tree-sha256', SOURCE_IDENTITY['source_tree_sha256'],
        '--output-dir', str(development_root),
        '--expected-world-size', '8',
        '--seed', '73012',
    ]
    print('Starting bounded synthetic production-backend XLA development campaign', flush=True)
    completed = subprocess.run(development_command, cwd=str(REPO), env=child_env, check=False)
    if completed.returncode:
        raise RuntimeError('XLA development campaign failed; retain rank failure receipts and Kaggle output.')
    development_path = development_root / 'aggregate.json'
    development_receipt = json.loads(development_path.read_text(encoding='utf-8'))
    assert development_receipt['status'] == 'PASS'
    assert development_receipt['production_training_authorized'] is False
    all_core_receipts['production_backend_xla_development'] = development_receipt
Path('/kaggle/working/signac_100m_all_core_canaries.json').write_text(
    json.dumps(all_core_receipts, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)
runtime_receipt_path = receipt_root / 'm102_primary' / 'aggregate.json'
qualification_receipt_path = (
    receipt_root / 'qualification' / 'm102_primary' / 'aggregate.json'
    if RUN_M102_QUALIFICATION else None
)
post_canary_report = build_report(
    target='tpu', runtime_receipt=runtime_receipt_path,
    qualification_receipt=qualification_receipt_path,
)
post_canary_report['source_identity'] = SOURCE_IDENTITY
post_canary_path = Path('/kaggle/working/signac_100m_post_canary_preflight.json')
post_canary_path.write_text(
    json.dumps(post_canary_report, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)
print('Post-canary review report:', json.dumps({
    'verdict': post_canary_report['verdict'],
    'canary_evidence': post_canary_report['gates']['target_runtime']['canary_evidence']['state'],
    'qualification_evidence': post_canary_report['gates']['target_runtime']['qualification_evidence']['state'],
    'blockers': post_canary_report['blockers'],
    'training_authorized': post_canary_report['training_authorized'],
}, indent=2))
assert post_canary_report['training_authorized'] is False


## Preserve outputs and stop

Save the timestamped `signac_100m_distributed/` rank and aggregate receipts, `signac_100m_static_preflight.json`, `signac_100m_all_core_canaries.json`, `signac_100m_post_canary_preflight.json`, runtime metadata, and the pinned notebook/source snapshot as Kaggle Output. If enabled, also retain `production_backend_xla_development/` with its two worker-group rank receipts and checkpoint store; this is a same-session synthetic resume check, not a durable Kaggle Output round-trip. The post-canary report hash-binds the raw M102 aggregate receipt for local review; it leaves the production runtime blocker in place. The all-core receipts show participation, synthetic update synchronization, state agreement, TPU device-memory counters, and per-worker host RSS observations after the final canary hashes when available. Host RSS is process-scoped and diagnostic; it does not measure production per-update hash staging. The smoke receipts are not the 20-update TPU peak-memory fit profile or a production throughput run.

The production trainer remains blocked until distributed numerical parity, at least 20 measured steady updates at the declared workload, measured peak-memory headroom, exact multi-process model/optimizer/per-rank-RNG/cursor restart, durable output round-trip, qualified data, Citadel evaluator readiness, and the fresh-seed baseline gates pass. Synthetic tokens never support a capability claim. Do not start the research run from these plumbing results alone.
